# Layer Function Characterization

What each layer does: logit effects, attn/MLP decomposition,
information change, role classification, and summary.

## Why This Matters

Layer function characterization characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_function_characterization import (
    layer_effect_on_logits, layer_attn_mlp_decomposition,
    layer_information_change, layer_role_classification,
    layer_characterization_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Layer Effect on Logits

How each layer changes the prediction landscape.

In [ ]:
result = layer_effect_on_logits(model, tokens, position=-1)
for p in result['per_layer']:
    flag = ' ← CHANGES PRED' if p['changes_prediction'] else ''
    print(f"  Layer {p['layer']}: MSE={p['logit_mse']:.4f}, "
          f"ΔH={p['entropy_change']:+.4f} (H: {p['pre_entropy']:.3f}→{p['post_entropy']:.3f}){flag}")

## Attention vs MLP Decomposition

Logit impact from each component.

In [ ]:
result = layer_attn_mlp_decomposition(model, tokens, position=-1)
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 20)
    mlp_bar = '█' * int((1 - p['attn_fraction']) * 20)
    print(f"  Layer {p['layer']}: attn={p['attn_logit_std']:.4f} {attn_bar}")
    print(f"           mlp ={p['mlp_logit_std']:.4f} {mlp_bar}")

## Information Change

New information vs amplification at each layer.

In [ ]:
result = layer_information_change(model, tokens)
for p in result['per_layer']:
    flag = ' [HIGH]' if p['is_high_change'] else ''
    print(f"  Layer {p['layer']}: angular={p['angular_change']:.4f}, "
          f"norm×={p['norm_change']:.4f}{flag}")

## Layer Role Classification

Refining, redirecting, amplifying, or maintaining?

In [ ]:
result = layer_role_classification(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: {p['role']:12s} | angular={p['angular_change']:.4f}, "
          f"ΔH={p['entropy_change']:+.4f}, norm×={p['norm_change']:.4f}")

## Characterization Summary

In [ ]:
result = layer_characterization_summary(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: {p['role']:12s} | ang={p['angular_change']:.4f}, "
          f"ΔH={p['entropy_change']:+.4f}, attn_frac={p['attn_fraction']:.2f}")